In [ ]:
# ============================================================
# Dialect-Controlled Machine Translation (DCMT)
# English → Sasak Multi-Dialect Translation
# Based on MarianMT with Dialect-Aware Gating Mechanism
#
# Author : Arik Aranta1, Arif Djunaidy, Nanik Suciati1
# Research :DCMT Architecture
# ============================================================

In [ ]:
# ============================================================
# Install Required Libraries
# ============================================================
!pip install -U \
  torch \
  transformers>=4.41.0 \
  accelerate \
  sentencepiece \
  sacrebleu \
  pandas \
  scikit-learn \
  tqdm


In [ ]:
# ============================================================
# Mount Google Drive
# Used for dataset access and model checkpoint storage
# ============================================================

from google.colab import drive
drive.mount("/content/drive")


Mounted at /content/drive


In [ ]:
# ============================================================
# Import Required Libraries
# ============================================================

import torch
import pandas as pd
from torch.utils.data import Dataset
from sklearn.model_selection import train_test_split

from transformers import (
    MarianMTModel,
    MarianTokenizer,
    Trainer,
    TrainingArguments,
    DataCollatorForSeq2Seq,
    EarlyStoppingCallback
)

# ============================================================
# Device Configuration
# ============================================================

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)


Device: cuda


In [ ]:
# ============================================================
# Load MarianMT Pretrained Model
# Base Model:
# Helsinki-NLP/opus-mt-en-id
# ============================================================

MODEL_NAME = "Helsinki-NLP/opus-mt-en-id"

tokenizer = MarianTokenizer.from_pretrained(MODEL_NAME)
model = MarianMTModel.from_pretrained(MODEL_NAME).to(device)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/42.0 [00:00<?, ?B/s]

source.spm:   0%|          | 0.00/796k [00:00<?, ?B/s]

target.spm:   0%|          | 0.00/801k [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

/usr/local/lib/python3.12/dist-packages/transformers/models/marian/tokenization_marian.py:175: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")


pytorch_model.bin:   0%|          | 0.00/291M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/293 [00:00<?, ?B/s]

In [ ]:
# ============================================================
# Dialect Dataset Class
# ============================================================
#
# This dataset class is designed for the proposed
# Dialect-Controlled Machine Translation (DCMT) model.
#
# Dataset Format:
# ------------------------------------------------------------
# | english | sasak | dialect |
# ------------------------------------------------------------
#
# Column Description:
# - english : Source sentence (English)
# - sasak   : Target sentence (Sasak)
# - dialect : Dialect label ID
#
# Dialect IDs:
# 0 = Dialect A
# 1 = Dialect B
# 2 = Dialect C
# 3 = Dialect D
# 4 = Dialect E
#
# Main Functions:
# 1. Tokenize source and target sequences
# 2. Convert text into token IDs
# 3. Attach dialect labels into each training batch
# 4. Prepare inputs for dialect-aware training
#
# Output:
# The dataset returns:
# - input_ids
# - attention_mask
# - labels
# - dialect_ids
#
# These outputs are later used by the proposed
# dialect-aware gating mechanism.
# ============================================================

class DialectDataset(Dataset):
    def __init__(self, df, tokenizer, max_len=128):
        """
        Initialize dataset.

        Parameters:
        --------------------------------------------------------
        df : pandas.DataFrame
            Dataset containing English, Sasak, and dialect columns

        tokenizer : MarianTokenizer
            Pretrained MarianMT tokenizer

        max_len : int
            Maximum token length for padding/truncation
        """

        self.df = df.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        """
        Return total number of samples.
        """
        return len(self.df)

    def __getitem__(self, idx):
         """
        Retrieve and preprocess a single dataset sample.

        Steps:
        --------------------------------------------------------
        1. Read source sentence (English)
        2. Read target sentence (Sasak)
        3. Read dialect label
        4. Tokenize source sentence
        5. Tokenize target sentence
        6. Attach dialect ID into training batch
        """

        # ====================================================
        # Source and Target Sentences
        # ====================================================

        src = self.df.loc[idx, "english"]
        tgt = self.df.loc[idx, "sasak"]
        # ====================================================
        # Dialect Label
        # Used for dialect-aware conditioning
        # ====================================================
        dialect_id = int(self.df.loc[idx, "dialect"])
        # ====================================================
        # Tokenize Source Sequence
        # ====================================================

        model_inputs = self.tokenizer(
            src,
            max_length=self.max_len,
            truncation=True,
            padding="max_length"
        )

        # ====================================================
        # Tokenize Target Sequence
        # ====================================================
        with self.tokenizer.as_target_tokenizer():
            labels = self.tokenizer(
                tgt,
                max_length=self.max_len,
                truncation=True,
                padding="max_length"
            )["input_ids"]

        # ====================================================
        # Attach Labels and Dialect IDs
        # ====================================================
        model_inputs["labels"] = labels
        # Dialect IDs are later used by the
        # dialect embedding and gating mechanism
        model_inputs["dialect_ids"] = dialect_id
        return model_inputs

In [ ]:
# ============================================================
# Proposed Dialect-Controlled Machine Translation (DCMT)
# ============================================================
#
# This module extends the standard MarianMT architecture
# by incorporating dialect-aware conditioning mechanisms.
#
# The proposed architecture introduces:
#
# 1. Dialect Embedding Layer
#    - Converts dialect IDs into dense vector representations
#
# 2. Dialect Projection Layer
#    - Projects dialect embeddings into the same hidden
#      representation space as encoder outputs
#
# 3. Gating Mechanism
#    - Dynamically controls how much dialect information
#      should influence each encoder hidden state
#
# 4. Dialect-Aware Fusion
#    - Combines contextual encoder representations with
#      dialect-specific representations
#
# ------------------------------------------------------------
# Core Fusion Equation:
#
# h̃_i = g_i ⊙ p_d + (1 - g_i) ⊙ h_i
#
# where:
# h_i   = encoder hidden representation
# p_d   = projected dialect embedding
# g_i   = learnable gating weight
# h̃_i  = dialect-aware hidden representation
#
# ============================================================
#
# Architecture Flow:
#
# Input Sentence
#       ↓
# Marian Encoder
#       ↓
# Encoder Hidden States (H)
#       ↓
# Dialect Embedding Lookup
#       ↓
# Dialect Projection
#       ↓
# Gating Mechanism
#       ↓
# Dialect-Aware Fusion
#       ↓
# Modified Encoder Representation (H̃)
#       ↓
# Marian Decoder
#       ↓
# Dialect-Specific Translation Output
#
# ============================================================
import torch.nn as nn
class DialectAwareMarianMT(nn.Module):
    def __init__(self, base_model, num_dialects=5):
       """
        Initialize the proposed DCMT architecture.

        Parameters:
        --------------------------------------------------------
        base_model : MarianMTModel
            Pretrained MarianMT backbone model

        num_dialects : int
            Total number of dialect labels
        """

        super().__init__()

        # ====================================================
        # Base MarianMT Model
        # ====================================================

        self.model = base_model
        # Hidden representation dimension (d_model)
        hidden_size = base_model.config.d_model
        # ====================================================
        # 1. Dialect Embedding Layer
        # ====================================================
        #
        # Converts dialect IDs into dense vector
        # representations.
        #
        # Example:
        # Dialect 0 → embedding vector
        # Dialect 1 → embedding vector
        #
        # Output Shape:
        # [batch_size, hidden_size]
        # ====================================================

        # 1. Embedding layer for dialect IDs
        self.dialect_embedding = nn.Embedding(num_dialects, hidden_size)
        # 2. Projection and gating mechanism
        self.dialect_proj = nn.Linear(hidden_size, hidden_size)
        self.gate = nn.Linear(hidden_size * 2, hidden_size)

    def forward(self, input_ids, attention_mask=None, labels=None, dialect_ids=None, **kwargs):
        # === 1. Standard encoder forward pass ===
        encoder_outputs = self.model.model.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask,
            return_dict=True
        )
        encoder_hidden = encoder_outputs.last_hidden_state  # [batch, seq_len, hidden_size]

        # === 2. Dialect Conditioning (Core Innovation) ===
        if dialect_ids is not None:
            # Get dialect embedding for the whole batch [batch, hidden_size]
            dialect_emb = self.dialect_embedding(dialect_ids)
            dialect_emb = self.dialect_proj(dialect_emb)
            # Expand to match encoder sequence length [batch, seq_len, hidden_size]
            dialect_emb = dialect_emb.unsqueeze(1).expand_as(encoder_hidden)

            # Learnable gating: decide how much dialect info to blend
            gate_input = torch.cat([encoder_hidden, dialect_emb], dim=-1)
            gate = torch.sigmoid(self.gate(gate_input))

            # Blend original encoder state with dialect information
            encoder_hidden = gate * dialect_emb + (1 - gate) * encoder_hidden

        # === 3. Decoder forward pass with conditioned encoder state ===
        outputs = self.model(
            encoder_outputs=(encoder_hidden,),
            attention_mask=attention_mask,
            labels=labels,
            return_dict=True
        )
        return outputs

# Initialize your dialect-aware model
base_model = MarianMTModel.from_pretrained(MODEL_NAME)
model = DialectAwareMarianMT(base_model, num_dialects=5).to(device)

In [ ]:
# ============================================================
# Dataset Loading and Training Preparation
# ============================================================
#
# This section prepares the multilingual multi-dialect
# dataset before model training.
#
# Processing Steps:
# ------------------------------------------------------------
# 1. Load cleaned English–Sasak dataset
# 2. Split dataset into training and validation sets
# 3. Create tokenized dataset objects
# 4. Initialize the proposed DCMT model
#
# Dataset Structure:
# ------------------------------------------------------------
# | english | sasak | dialect |
# ------------------------------------------------------------
#
# Example:
#
# english : "How are you?"
# sasak   : "Apa kabar?"
# dialect : 2
#
# ============================================================


# ============================================================
# A. Load Dataset
# ============================================================
#
# Load the cleaned English-to-Sasak multi-dialect dataset.
#
# Dataset contains:
# - Source sentence (English)
# - Target sentence (Sasak)
# - Dialect label
#
# File Format:
# CSV (comma separated)
# ============================================================
# 🔴 THESE MUST EXIST BEFORE trainer = DialectTrainer(...)

# A) Load and prepare your data
df = pd.read_csv("/content/V3_15_1_26_sasak_dialects_cleaned.csv", sep=',') # Changed sep='\t' to sep=','
# df = df.rename(columns={'dialect_id': 'dialect'})  # Match dataset class - this line is likely not needed if sep=',' works correctly

# B) Split into train/validation
df_train, df_val = train_test_split(df, test_size=0.1, random_state=42)

# C) Create tokenized datasets
train_dataset = DialectDataset(df_train, tokenizer)   # ← Must exist
val_dataset = DialectDataset(df_val, tokenizer)       # ← Must exist

# D) Initialize your dialect-aware model
base_model = MarianMTModel.from_pretrained(MODEL_NAME)
model = DialectAwareMarianMT(base_model, num_dialects=5).to(device)

In [ ]:
# ============================================================
# Custom Dataset Parsing Utility
# ============================================================
#
# This section provides a manual parsing mechanism for the
# English–Sasak multi-dialect dataset.
#
# The parser is designed to handle multiple file formatting
# variations that may occur in manually collected datasets,
# including:
#
# 1. Tab-separated values (.tsv)
# 2. Standard comma-separated values (.csv)
# 3. Quoted CSV formats
#
# The parser converts raw dataset files into a structured
# pandas DataFrame compatible with the proposed DCMT model.
#
# ============================================================
import csv

# ============================================================
# Dataset Parsing Function
# ============================================================
#
# Input:
# ------------------------------------------------------------
# filepath : str
#     Path to the dataset file
#
# Output:
# ------------------------------------------------------------
# pandas.DataFrame containing:
#
# | english | sasak | dialect |
#
# ============================================================
def parse_sasak_dataset(filepath):
"""
    Parse English–Sasak multi-dialect dataset.

    The function automatically detects several
    separator formats and converts the dataset
    into structured tabular format.
    """

    # ========================================================
    # Store Parsed Samples
    # ========================================================
    data = []

    with open(filepath, 'r', encoding='utf-8') as f:
        # Skip header if needed
        header = f.readline().strip()
        print(f"Header: {header}")

        for line_num, line in enumerate(f, 1):
            line = line.strip()
            if not line:
                continue

            # Try different splitting strategies
            if '\t' in line:
                # Tab-separated
                parts = line.split('\t')
            elif '","' in line:
                # CSV with quotes
                parts = [p.strip('"') for p in line.split('","')]
            else:
                # Simple comma-separated (risky due to commas in text)
                parts = line.split(',', 2)  # Split into max 3 parts

            if len(parts) >= 3:
                english = parts[0].strip()
                sasak = parts[1].strip()
                dialect = parts[2].strip()
                data.append({
                    'english': english,
                    'sasak': sasak,
                    'dialect': int(dialect) if dialect.isdigit() else dialect
                })
            else:
                print(f"Warning: Line {line_num} has {len(parts)} parts: {line[:50]}...")

    return pd.DataFrame(data)

# Use this instead of pd.read_csv
df = parse_sasak_dataset("/content/V3_15_1_26_sasak_dialects_cleaned.csv")
print(f"Parsed {len(df)} rows")
print(f"Columns: {list(df.columns)}")
print("\nFirst sample:")
print(f"English: {df['english'].iloc[0][:50]}...") # Corrected f-string and .iloc usage
print(f"Sasak: {df['sasak'].iloc[0][:50]}...")   # Corrected f-string and .iloc usage
print(f"Dialect: {df['dialect'].iloc[0]}")      # Corrected f-string and .iloc usage

In [ ]:
# ============================================================
# Load Cleaned Multi-Dialect Dataset
# ============================================================
#
# This section loads the finalized English–Sasak
# multi-dialect dataset using pandas.
#
# Dataset Format:
# ------------------------------------------------------------
# | english | sasak | dialect |
#
# Column Description:
# - english : Source language sentence
# - sasak   : Target translation sentence
# - dialect : Dialect label identifier
#
# File Format:
# CSV (Comma-Separated Values)
#
# ============================================================

# ============================================================
# Read Dataset File
# ============================================================

df = pd.read_csv(
    "/content/V3_15_1_26_sasak_dialects_cleaned.csv",
    sep=',' # Changed from '\t' to ','
)

print(f"Loaded {len(df)} rows")
print(df.head())

Loaded 10000 rows
                                             english  \
0   Do you have any siblings? No, I'm an only child.   
1    Are these your books? No, they're not my books.   
2  Are you going to Tom's party? I'm still not sure.   
3   Do you have any siblings? No, I'm an only child.   
4    Are these your books? No, they're not my books.   

                                               sasak  dialect  
0  apakaha dea memilikia semetona kandung?a tidak...        4  
1  apakaha nea. bukua -bukua anda? tidaka, tiea b...        4  
2  apakaha taoqma yaqa laloa joka pestaa tom? aku...        4  
3  apakahe dee memilikie semetone kandung?e tidak...        1  
4  apakahe nee. bukue -bukue anda? tidake, tiee b...        1  


In [ ]:
# ============================================================
# Final Dataset Quality Validation
# ============================================================
#
# This section performs a comprehensive quality check
# before the training process.
#
# Validation Objectives:
# ------------------------------------------------------------
# 1. Verify total dataset samples
# 2. Check column data types
# 3. Analyze dialect distribution balance
# 4. Detect missing values
# 5. Inspect sample translations from each dialect
#
# This validation step is important to ensure:
# - Data consistency
# - Balanced dialect representation
# - Clean training input
# - Reliable model learning
#
# ============================================================
print("🔍 FINAL DATA QUALITY CHECK")
print(f"Total samples: {len(df)}")
print(f"Data types:\n{df.dtypes}")
print(f"\nDialect distribution:")
dialect_counts = df['dialect'].value_counts().sort_index()
for dial_id, count in dialect_counts.items():
    print(f"  Dialect {dial_id}: {count} samples ({count/len(df)*100:.1f}%)")

print(f"\nMissing values:")
print(df.isnull().sum())

# Verify a few samples
print(f"\n📝 Sample verification (first 2 samples per dialect):")
for dial_id in sorted(df['dialect'].unique()):
    print(f"\n=== Dialect {dial_id} ===")
    samples = df[df['dialect'] == dial_id].head(2)
    for idx, row in samples.iterrows():
        print(f"English: {row['english'][:50]}...")
        print(f"Sasak:   {row['sasak'][:50]}...")
        print()

🔍 FINAL DATA QUALITY CHECK
Total samples: 10000
Data types:
english    object
sasak      object
dialect     int64
dtype: object

Dialect distribution:
  Dialect 0: 2000 samples (20.0%)
  Dialect 1: 2000 samples (20.0%)
  Dialect 2: 2000 samples (20.0%)
  Dialect 3: 2000 samples (20.0%)
  Dialect 4: 2000 samples (20.0%)

Missing values:
english    0
sasak      0
dialect    0
dtype: int64

📝 Sample verification (first 2 samples per dialect):

=== Dialect 0 ===
English: Do you have any siblings? No, I'm an only child....
Sasak:   apakahi dei memilikii semetoni kandung?i tidaki, T...

English: Are these your books? No, they're not my books....
Sasak:   apakahi nei. bukui -bukui anda? tidaki, tiei bukan...


=== Dialect 1 ===
English: Do you have any siblings? No, I'm an only child....
Sasak:   apakahe dee memilikie semetone kandung?e tidake, T...

English: Are these your books? No, they're not my books....
Sasak:   apakahe nee. bukue -bukue anda? tidake, tiee bukan...


=== Dialect 2 ===
E

In [ ]:
# AFTER loading df, BEFORE creating datasets
df = pd.read_csv("/content/V3_15_1_26_sasak_dialects_cleaned.csv", sep='\t')

print("🔍 ACTUAL COLUMNS IN YOUR DATASET:")
print(f"Columns: {list(df.columns)}")
print(f"First few rows:\n{df.head()}")

🔍 ACTUAL COLUMNS IN YOUR DATASET:
Columns: ['english,sasak,dialect']
First few rows:
                               english,sasak,dialect
0  Do you have any siblings? No, I'm an only chil...
1  Are these your books? No, they're not my books...
2  Are you going to Tom's party? I'm still not su...
3  Do you have any siblings? No, I'm an only chil...
4  Are these your books? No, they're not my books...


In [ ]:
# ============================================================
# Dataset Validation Function
# ============================================================
#
# This function performs a comprehensive validation process
# for the English–Sasak multi-dialect dataset before model
# training.
#
# Validation Objectives:
# ------------------------------------------------------------
# 1. Verify dataset structure
# 2. Check dataset columns
# 3. Validate dialect label distribution
# 4. Detect missing values
# 5. Inspect translation consistency
#
# The validation step ensures that the dataset is:
# - correctly formatted
# - balanced across dialects
# - free from null values
# - suitable for DCMT training
#
# ============================================================

def validate_dataset(df_path):
    # Fix 1: Use comma as separator
    df = pd.read_csv(df_path, sep=',')  # Use comma separator for your format

    print("🔍 DATASET VALIDATION")
    print(f"Total samples: {len(df)}")
    print(f"Columns: {list(df.columns)}")

    # Fix 2: Rename 'dialect_id' to 'dialect' for consistency with subsequent code
    # This assumes the CSV file, once correctly parsed, has a 'dialect_id' column
    # and that the rest of the function expects a 'dialect' column.
    if 'dialect_id' in df.columns and 'dialect' not in df.columns:
        df = df.rename(columns={'dialect_id': 'dialect'})

    # Check dialect distribution
    print("\n📊 DIALECT DISTRIBUTION:")
    # Ensure 'dialect' column exists before trying to access it
    if 'dialect' in df.columns:
        dialect_counts = df['dialect'].value_counts().sort_index()
        for dial_id, count in dialect_counts.items():
            print(f"  Dialect {dial_id}: {count} samples")
    else:
        print("  WARNING: 'dialect' column not found for distribution check.")

    # Check for missing values
    print("\n✅ MISSING VALUES CHECK:")
    missing = df.isnull().sum()
    if missing.sum() == 0:
        print("  No missing values - Excellent!")
    else:
        print(f"  WARNING: Missing values found:\n{missing}")

    # Check sample consistency
    print("\n🔤 SAMPLE CHECK (first example per dialect):")
    # After renaming, 'dialect' should be used consistently
    if 'dialect' in df.columns:
        for dial_id in sorted(df['dialect'].unique()):
            sample = df[df['dialect'] == dial_id].iloc[0]
            print(f"\n  Dialect {dial_id}:")
            print(f"    English: {sample['english'][:50]}...")
            print(f"    Sasak: {sample['sasak'][:50]}...")
    else:
        print("  WARNING: 'dialect' column not found for sample check.")

    return df

# Run validation
df = validate_dataset("/content/V3_15_1_26_sasak_dialects_cleaned.csv")

🔍 DATASET VALIDATION
Total samples: 10000
Columns: ['english', 'sasak', 'dialect']

📊 DIALECT DISTRIBUTION:
  Dialect 0: 2000 samples
  Dialect 1: 2000 samples
  Dialect 2: 2000 samples
  Dialect 3: 2000 samples
  Dialect 4: 2000 samples

✅ MISSING VALUES CHECK:
  No missing values - Excellent!

🔤 SAMPLE CHECK (first example per dialect):

  Dialect 0:
    English: Do you have any siblings? No, I'm an only child....
    Sasak: apakahi dei memilikii semetoni kandung?i tidaki, T...

  Dialect 1:
    English: Do you have any siblings? No, I'm an only child....
    Sasak: apakahe dee memilikie semetone kandung?e tidake, T...

  Dialect 2:
    English: Do you have any siblings? No, I'm an only child....
    Sasak: apakaho deo memilikio semetono kandung?o tidako, T...

  Dialect 3:
    English: Do you have any siblings? No, I'm an only child....
    Sasak: apakahu deu memilikiu semetonu kandung?u tidaku, T...

  Dialect 4:
    English: Do you have any siblings? No, I'm an only child....
    

In [ ]:
# ============================================================
# Stratified Train–Validation Split
# ============================================================
#
# This section divides the validated dataset into:
#
# 1. Training Dataset
# 2. Validation Dataset
#
# Purpose:
# ------------------------------------------------------------
# - Training dataset:
#   Used to optimize model parameters.
#
# - Validation dataset:
#   Used to evaluate model performance during training.
#
# Stratified Splitting:
# ------------------------------------------------------------
# The split uses stratification based on dialect labels
# to preserve balanced dialect distribution across:
#
# - training data
# - validation data
#
# This is important for multi-dialect learning to avoid:
# - dialect imbalance
# - biased learning
# - overfitting to dominant dialects
#
# ============================================================


# ============================================================
# Load Validated Dataset
# ============================================================
#
# Ensure the dataset contains:
# - english
# - sasak
# - dialect
#
# ============================================================

df = pd.read_csv("/content/V3_15_1_26_sasak_dialects_cleaned.csv", sep=',')

# 1. Split into train/validation (DIFFERENT sets!)
df_train, df_val = train_test_split(
    df,
    test_size=0.1,           # 10% for validation
    random_state=42,         # For reproducibility
    stratify=df['dialect']   # Keep dialect balance
)

print(f"Training: {len(df_train)} samples")
print(f"Validation: {len(df_val)} samples")

Training: 9000 samples
Validation: 1000 samples


In [ ]:
# ============================================================
# Custom Dialect-Aware Trainer
# ============================================================
#
# This class extends the HuggingFace Trainer class
# to support the proposed Dialect-Controlled Machine
# Translation (DCMT) architecture.
#
# Purpose:
# ------------------------------------------------------------
# The default HuggingFace Trainer does not automatically
# process additional dialect labels during training.
#
# Therefore, this custom trainer is implemented to:
#
# 1. Extract dialect labels from each batch
# 2. Pass dialect IDs into the proposed DCMT model
# 3. Enable dialect-aware conditioning during training
# 4. Compute translation loss for optimization
#
# ============================================================
class DialectTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        # Extract dialect_ids before passing to model
        dialect_ids = inputs.pop("dialect_ids", None)
        # Forward pass with dialect conditioning
        outputs = model(**inputs, dialect_ids=dialect_ids)
        loss = outputs.loss
        return (loss, outputs) if return_outputs else loss

In [ ]:
# ============================================================
# Training Configuration
# ============================================================
#
# This section defines the training configuration for the
# proposed Dialect-Controlled Machine Translation (DCMT)
# model using HuggingFace TrainingArguments.
#
# Training Objectives:
# ------------------------------------------------------------
# 1. Optimize translation performance
# 2. Monitor validation loss during training
# 3. Save the best-performing model
# 4. Reduce memory usage using mixed precision
# 5. Prevent excessive checkpoint storage
#
# ============================================================


# ============================================================
# Output Directory
# ============================================================
#
# Directory used to store:
# - trained checkpoints
# - best model
# - training logs
#
# ============================================================

OUTPUT_DIR = "/content/drive/MyDrive/PhD_marian_dialect_model"

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    eval_strategy="epoch",
    save_strategy="epoch",
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    learning_rate=3e-5,
    num_train_epochs=20,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    logging_steps=100,
    report_to="none",
    fp16=torch.cuda.is_available(),
    save_safetensors=False # Added to resolve RuntimeError with shared tensors
)

In [ ]:
# ============================================================
# Data Collator and Trainer Initialization
# ============================================================
#
# This section prepares the training pipeline for the
# proposed Dialect-Controlled Machine Translation (DCMT)
# architecture.
#
# Main Components:
# ------------------------------------------------------------
# 1. Data Collator
#    Handles dynamic batch preparation for sequence-to-
#    sequence translation tasks.
#
# 2. Custom DialectTrainer
#    Executes dialect-aware training using:
#    - encoder representations
#    - dialect embeddings
#    - gating mechanism
#
# 3. Early Stopping
#    Prevents overfitting by stopping training when
#    validation loss no longer improves.
#
# ============================================================


# ============================================================
# Sequence-to-Sequence Data Collator
# ============================================================
#
# The DataCollatorForSeq2Seq dynamically prepares batches
# during training and validation.
#
# Functions:
# ------------------------------------------------------------
# - pad variable-length sequences
# - align input and target tokens
# - optimize batch processing efficiency
#
# tokenizer :
#     Marian tokenizer used for tokenization
#
# model :
#     Base MarianMT architecture
#
# ============================================================

# Keep your data_collator as is
data_collator = DataCollatorForSeq2Seq(tokenizer, model=base_model)

# Instantiate trainer with dialect-aware components
trainer = DialectTrainer(
    model=model,  # Your new DialectAwareMarianMT
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

In [ ]:
trainer.train()  # This will use dialect_ids from your datasets

In [ ]:
# ============================================================
# Google Drive Mounting
# ============================================================
#
# This section connects Google Colab to Google Drive
# for persistent storage access.
#
# Purpose:
# ------------------------------------------------------------
# The mounted Google Drive is used to:
#
# 1. Store trained model checkpoints
# 2. Save training logs and outputs
# 3. Access dataset files
# 4. Prevent data loss after Colab session termination
#
# Mount Path:
# ------------------------------------------------------------
# /content/drive
#
# ============================================================from google.colab import drive
drive.mount('/content/drive')